# Yield Curve Forecasting using PCA

### Market Trading Algorithmic Signals — Banking-AI

This notebook explains how traders analyze the relationship between different interest rates using Principal Component Analysis (PCA):

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of market trading and algorithmic signals terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Describe the information-extraction task and its relevance to market trading and algorithmic signals.
- Implement a rule-based extraction pipeline and evaluate accuracy on synthetic examples.
- Assess limitations of rule-based extraction and when ML-based approaches would be more appropriate.

**Estimated time:** 35–45 minutes

**Collection context:** Volatility forecasting, trading signals, price prediction, and quantitative market analysis.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** The "Yield Curve" shows the interest rates for borrowing money over different periods of time. Normally, it's cheaper to borrow for 1 month than for 30 years. However, all these rates (1m, 2y, 10y, 30y) tend to move together. Instead of tracking 10 different rates, traders use **PCA** to condense them into 3 simple factors. This makes it much easier to predict how the entire economy is shifting.

**Input data used:** Daily interest rates for 1m, 3m, 6m, 1y, 2y, 5y, 10y, 20y, 30y.

**What we analyze:** The "Factors" that drive 99% of yield curve movement.

**ML method used:** Principal Component Analysis (PCA).

**Learning goal:** Understand dimensionality reduction in finance.

**Primary stakeholders:** quantitative analysts, portfolio managers, risk officers, and trading desk supervisors.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Yield Curve Data

We simulate daily moves for various interest rate maturities.

In [ ]:
n_days = 1000
maturities = ['1M', '3M', '6M', '1Y', '2Y', '5Y', '10Y', '20Y', '30Y']
n_mats = len(maturities)

# 1. Simulate 3 underlying factors: Level (Parallel shift), Slope (Tilt), Curvature (Flex)
level = np.cumsum(RNG.normal(0, 0.05, n_days)) + 2.0
slope = np.cumsum(RNG.normal(0, 0.02, n_days)) + 1.0
curvature = np.cumsum(RNG.normal(0, 0.01, n_days))

# 2. Map these factors to the maturities
data = np.zeros((n_days, n_mats))
for i, m in enumerate([1/12, 0.25, 0.5, 1, 2, 5, 10, 20, 30]):
    data[:, i] = level + (slope * (i / n_mats)) + (curvature * np.sin(i / 2)) + RNG.normal(0, 0.01, n_days)

df = pd.DataFrame(data, columns=maturities)

plt.figure(figsize=(10, 6))
for m in ['1M', '2Y', '10Y', '30Y']:
    plt.plot(df[m], label=m)
plt.title('Daily Interest Rates for Different Maturities')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

For this workflow, the data prepared in Step 3 is used directly as input features.

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: manual extraction (no automation) ---
print("Baseline: without automation, each document must be read and keyed manually.")
print("Any automated approach that achieves >80% field accuracy saves significant time.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

We use PCA to see how many components are needed to explain the movement.

In [ ]:
# PCA works best on standardized data (mean=0, std=1)
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

pca = PCA(n_components=3)
pca_features = pca.fit_transform(df_scaled)

print(f"Explained Variance by 3 components: {sum(pca.explained_variance_ratio_)*100:.2f}%")

In [ ]:
loadings = pd.DataFrame(pca.components_.T, 
                        columns=['PC1 (Level)', 'PC2 (Slope)', 'PC3 (Curvature)'], 
                        index=maturities)

plt.figure(figsize=(10, 6))
plt.plot(loadings['PC1 (Level)'], marker='o', label='PC1: Parallel Shift')
plt.plot(loadings['PC2 (Slope)'], marker='o', label='PC2: Curve Tilt')
plt.plot(loadings['PC3 (Curvature)'], marker='o', label='PC3: Curve Flex')
plt.title('Yield Curve PCA Components (Loadings)')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Line chart titled "Yield Curve PCA Components (Loadings)". The chart shows temporal trends and the alignment between predicted and observed values.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(pca_features[:, 0], label='Level Factor')
plt.plot(pca_features[:, 1], label='Slope Factor')
plt.title('Movement of Yield Curve Factors Over Time')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Line chart titled "Movement of Yield Curve Factors Over Time". The chart shows temporal trends and the alignment between predicted and observed values.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary:**
- **PC1 (Level)** explains ~90%+ of the variance. It represents all interest rates moving up or down together (e.g., when the Fed raises rates).
- **PC2 (Slope)** explains the curve getting steeper or flatter (e.g., long-term growth expectations).
- **PC3 (Curvature)** represents the middle of the curve moving differently than the ends.

**Market Context:** Central Banks and Hedge Funds use PCA to spot "Rich/Cheap" segments of the curve. If the 5-year rate is much higher than what the Level/Slope/Curvature factors predict, it might be an "Alpha" opportunity to buy that bond, expecting its yield to fall back in line with the rest of the curve.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: process a new document ---
print("Operational demonstration — field extraction:\n")
new_doc = "Invoice #2055. Vendor: Wayne Enterprises. Date: 2024-06-15. Total: $8750.00"
result = extract_info(new_doc)
print(f"  Raw: {new_doc}")
print(f"  Vendor: {result[0]}  Date: {result[1]}  Amount: ${result[2]:,.2f}")
print("\nExtracted fields would auto-populate the accounts-payable system.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end market trading and algorithmic signals workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Algorithmic trading models can amplify market volatility and systemic risk if deployed without safeguards.
- Backtested performance often overstates real-world returns due to look-ahead bias and transaction costs.
- Automated trading decisions must include human oversight and circuit breakers.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in market trading and algorithmic signals?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.